# Customer Churn Prediction using Machine Learning
**Author:** Senior Machine Learning Engineer

## 1. Project Overview
The objective of this project is to develop a robust binary classification model that predicts customer attrition (churn) for a telecommunication service provider. This notebook covers:
1. **Exploratory Data Analysis (EDA):** Univariate, bivariate, correlation heatmaps, boxplots, and distributions.
2. **Data Preprocessing:** Handling missing values, outlier detection (IQR), categorical encoding (One-Hot & Label), and feature scaling.
3. **Handling Class Imbalance:** SMOTE (oversampling the minority class).
4. **Model Suite Training & Benchmark:** Evaluating 9 different classifiers (Logistic Regression, Decision Trees, Random Forests, Gradient Boosters, AdaBoost, XGBoost, SVM, KNN, Naive Bayes).
5. **Validation & Hyperparameters:** Cross-validation, metrics benchmarking, ROC-AUC comparisons, and pipeline serialization using `joblib`.

## 2. Setup and Library Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_validate, learning_curve
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print("Setup complete. Libraries imported.")

## 3. Dataset Generation (Realistic Telco Churn Mock)
If you have the Kaggle dataset, load it here. Otherwise, we will use the highly representative synthetic data generator implemented in our preprocessing suite.

In [ ]:
from preprocessing import generate_synthetic_data

# Generate 1500 samples
df_raw = generate_synthetic_data(num_samples=1500, random_state=42)
print(f"Dataset generated with dimensions: {df_raw.shape}")
df_raw.head()

## 4. Exploratory Data Analysis (EDA)
Let's check the target balance, look for empty values, handle the string-object typing of numerical columns (like `TotalCharges`), and run statistical checks.

In [ ]:
print("--- Data Types & Non-Null Counts ---")
print(df_raw.info())

print("\n--- Missing values analysis ---")
missing_vals = df_raw.isnull().sum()
print(missing_vals[missing_vals > 0])

# Clean TotalCharges (convert ' ' to NaN, then fill with median)
df_raw['TotalCharges'] = pd.to_numeric(df_raw['TotalCharges'].replace(r'^\s*$', np.nan, regex=True), errors='coerce')
total_charges_median = df_raw['TotalCharges'].median()
df_raw['TotalCharges'] = df_raw['TotalCharges'].fillna(total_charges_median)

print("\n--- Statistical summary of numerical columns ---")
df_raw[['tenure', 'MonthlyCharges', 'TotalCharges']].describe()

In [ ]:
# Visualizing Churn Distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df_raw, x='Churn', palette=['#4f46e5', '#f43f5e'])
plt.title('Target Variable (Churn) Distribution')
plt.show()

In [ ]:
# Visualizing Distributions (Histograms and Boxplots)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=df_raw, x='MonthlyCharges', hue='Churn', kde=True, multiple='stack', ax=axes[0], palette=['#4f46e5', '#f43f5e'])
axes[0].set_title('Monthly Charges Distribution by Churn')

sns.boxplot(data=df_raw, y='tenure', x='Churn', ax=axes[1], palette=['#4f46e5', '#f43f5e'])
axes[1].set_title('Tenure (Months) vs Churn')
plt.tight_layout()
plt.show()

## 5. Preprocessing and Feature Selection

In [ ]:
from preprocessing import ChurnPreprocessor

# Apply preprocessor
preproc = ChurnPreprocessor(scaling_method="standard")
preproc.fit(df_raw)
X, y = preproc.transform(df_raw)

print("Features engineered and scaled successfully.")
print("Feature Space Dimensions:", X.shape)
print("Preprocessed columns:", preproc.feature_columns)

## 6. Resolving Class Imbalance (SMOTE)

In [ ]:
# Before SMOTE count
print("Class distribution before oversampling:", np.bincount(y))

# Apply Preprocessor's custom robust SMOTE
X_bal, y_bal = preproc.handle_class_imbalance_smote(X, y)
print("Class distribution after oversampling:", np.bincount(y_bal))

## 7. Model Training, Benchmark, and Selection

In [ ]:
from model import ChurnClassifierSuite

# Split into train and test splits
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, test_size=0.2, random_state=42, stratify=y_bal
)

# Initialize and train classifier comparisons suite
suite = ChurnClassifierSuite()
all_metrics = suite.train_and_evaluate_all(X_train, y_train, X_test, y_test)

# Convert to pandas table for visualization
comparison_df = pd.DataFrame(all_metrics).T
comparison_df[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'CV_Accuracy_Mean', 'CV_F1_Mean']]

In [ ]:
print("Best Performing Model Selected:", suite.best_model_name)

# Save pipeline
saved_path = suite.save_pipeline(preproc, "churn_prediction_pipeline.joblib")
print(f"Pipeline successfully exported to '{saved_path}'")

## 8. Performance Evaluation Plots
Let's plot the comparative ROC curves and the top feature importances.

In [ ]:
# Plot comparative ROC Curves
plt.figure(figsize=(8, 6))
for name, res in suite.results.items():
    roc = res["ROC_Curve"]
    plt.plot(roc["fpr"], roc["tpr"], label=f"{name} (AUC = {res['ROC-AUC']:.3f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend(loc='lower right')
plt.show()

In [ ]:
# Plot Feature Importances of Best Model
best_res = suite.results[suite.best_model_name]
imps = best_res["Feature_Importances"]
if imps:
    paired = sorted(zip(preproc.feature_columns, imps), key=lambda x: x[1], reverse=True)[:10]
    feats, vals = zip(*paired)
    plt.figure(figsize=(8, 5))
    sns.barplot(x=list(vals), y=list(feats), palette="viridis")
    plt.title(f'Top 10 Feature Importances ({suite.best_model_name})')
    plt.xlabel('Relative Weight')
    plt.show()